In [7]:
import nfl_data_py as nfl
import pandas as pd

# Load one season of play-by-play data
pbp = nfl.import_pbp_data([2025])

print(f"Shape: {pbp.shape}")
print(f"\nColumns ({len(pbp.columns)}):")
print(list(pbp.columns))

2025 done.
Downcasting floats.
Shape: (48771, 397)

Columns (397):
['play_id', 'game_id', 'old_game_id_x', 'home_team', 'away_team', 'season_type', 'week', 'posteam', 'posteam_type', 'defteam', 'side_of_field', 'yardline_100', 'game_date', 'quarter_seconds_remaining', 'half_seconds_remaining', 'game_seconds_remaining', 'game_half', 'quarter_end', 'drive', 'sp', 'qtr', 'down', 'goal_to_go', 'time', 'yrdln', 'ydstogo', 'ydsnet', 'desc', 'play_type', 'yards_gained', 'shotgun', 'no_huddle', 'qb_dropback', 'qb_kneel', 'qb_spike', 'qb_scramble', 'pass_length', 'pass_location', 'air_yards', 'yards_after_catch', 'run_location', 'run_gap', 'field_goal_result', 'kick_distance', 'extra_point_result', 'two_point_conv_result', 'home_timeouts_remaining', 'away_timeouts_remaining', 'timeout', 'timeout_team', 'td_team', 'td_player_name', 'td_player_id', 'posteam_timeouts_remaining', 'defteam_timeouts_remaining', 'total_home_score', 'total_away_score', 'posteam_score', 'defteam_score', 'score_different

In [8]:
# Look at the columns most relevant to models
key_cols = ['play_type', 'down', 'ydstogo', 'yardline_100', 
            'epa', 'wp', 'wpa', 'score_differential',
            'offense_formation', 'defense_coverage_type',
            'was_pressure', 'xpass', 'pass_oe', 'cpoe']

print(pbp[key_cols].describe())
print("\nNull counts:")
print(pbp[key_cols].isnull().sum())

               down       ydstogo  yardline_100           epa            wp  \
count  40778.000000  48771.000000  45223.000000  48201.000000  48486.000000   
mean       2.007651      7.046011     47.139088      0.013139      0.509186   
std        1.006315      4.866811     23.757689      1.258956      0.289186   
min        1.000000      0.000000      1.000000    -12.658112      0.000029   
25%        1.000000      3.000000     30.000000     -0.551978      0.285741   
50%        2.000000      9.000000     48.000000      0.000000      0.519305   
75%        3.000000     10.000000     66.000000      0.551108      0.731358   
max        4.000000     36.000000     99.000000      7.967405      0.999972   

                wpa  score_differential  was_pressure         xpass  \
count  48033.000000        46042.000000  45175.000000  37054.000000   
mean       0.001242           -1.352352      0.146807      0.627238   
std        0.041712           10.313481      0.353917      0.242487   
min 

In [9]:
# Filter to run/pass plays only — core analytical dataset
pbp_plays = pbp[pbp['play_type'].isin(['run', 'pass'])].copy()

print(f"Total plays: {len(pbp)}")
print(f"Run/pass plays: {len(pbp_plays)}")
print(f"\nPlay type breakdown:")
print(pbp_plays['play_type'].value_counts())

print(f"\nAverage EPA by play type:")
print(pbp_plays.groupby('play_type')['epa'].mean().round(3))

print(f"\nAverage EPA by down:")
print(pbp_plays.groupby('down')['epa'].mean().round(3))

Total plays: 48771
Run/pass plays: 34628

Play type breakdown:
pass    19735
run     14893
Name: play_type, dtype: int64

Average EPA by play type:
play_type
pass    0.014
run    -0.007
Name: epa, dtype: float32

Average EPA by down:
down
1.0    0.009
2.0    0.019
3.0   -0.054
4.0    0.216
Name: epa, dtype: float32


In [10]:
# EPA by down and distance situation
pbp_plays['situation'] = pbp_plays.apply(
    lambda x: 'short' if x['ydstogo'] <= 3 
    else 'medium' if x['ydstogo'] <= 7 
    else 'long', axis=1
)

print("EPA by down and distance situation:")
print(pbp_plays.groupby(['down', 'situation'])['epa'].mean().round(3).unstack())

print("\n--- Top 10 formations by avg EPA (min 50 plays) ---")
formation_epa = (pbp_plays[pbp_plays['offense_formation'].notna()]
    .groupby('offense_formation')
    .agg(plays=('epa', 'count'), avg_epa=('epa', 'mean'))
    .query('plays >= 50')
    .sort_values('avg_epa', ascending=False)
    .round(3))
print(formation_epa)

print("\n--- Pass rate by down ---")
pbp_plays['is_pass'] = (pbp_plays['play_type'] == 'pass').astype(int)
print(pbp_plays.groupby('down')['is_pass'].mean().round(3))

EPA by down and distance situation:
situation   long  medium  short
down                           
1.0        0.007   0.054  0.041
2.0        0.003   0.038  0.031
3.0       -0.106  -0.049  0.010
4.0       -0.665   0.230  0.391

--- Top 10 formations by avg EPA (min 50 plays) ---
                   plays  avg_epa
offense_formation                
UNDER CENTER       11725    0.013
SHOTGUN            21196    0.002
PISTOL              1649   -0.018

--- Pass rate by down ---
down
1.0    0.472
2.0    0.588
3.0    0.734
4.0    0.640
Name: is_pass, dtype: float64


In [11]:
print("Personnel groupings (min 100 plays):")
personnel_epa = (pbp_plays[pbp_plays['offense_personnel'].notna()]
    .groupby('offense_personnel')
    .agg(plays=('epa', 'count'), avg_epa=('epa', 'mean'))
    .query('plays >= 100')
    .sort_values('plays', ascending=False)
    .round(3))
print(personnel_epa)

Personnel groupings (min 100 plays):
                                             plays  avg_epa
offense_personnel                                          
1 C, 2 G, 1 QB, 1 RB, 2 T, 1 TE, 3 WR        14533    0.003
1 C, 2 G, 1 QB, 1 RB, 2 T, 2 TE, 2 WR         5792   -0.001
1 C, 1 G, 1 QB, 1 RB, 3 T, 1 TE, 3 WR         1780    0.106
1 C, 2 G, 1 QB, 1 RB, 2 T, 3 TE, 1 WR         1437    0.052
2 C, 1 G, 1 QB, 1 RB, 2 T, 1 TE, 3 WR         1270   -0.037
1 C, 1 FB, 2 G, 1 QB, 1 RB, 2 T, 1 TE, 2 WR   1086    0.015
3 G, 1 QB, 1 RB, 2 T, 1 TE, 3 WR               760    0.027
1 C, 3 G, 1 QB, 1 RB, 1 T, 1 TE, 3 WR          727   -0.043
2 C, 1 G, 1 QB, 1 RB, 2 T, 2 TE, 2 WR          556   -0.057
1 C, 1 FB, 2 G, 1 QB, 1 RB, 2 T, 2 TE, 1 WR    534    0.074
1 C, 2 G, 1 QB, 2 RB, 2 T, 1 TE, 2 WR          480   -0.089
1 C, 1 G, 1 QB, 1 RB, 3 T, 2 TE, 2 WR          474    0.074
3 G, 1 QB, 1 RB, 2 T, 2 TE, 2 WR               352   -0.020
1 C, 1 FB, 1 G, 1 QB, 1 RB, 3 T, 1 TE, 2 WR    300    0.093
1 C